# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule in plain words

I prioritize pages that are visible enough for an improvement to matter and that show at least one reason for review: they have gone a long time without an update, or their CTR is weak relative to pages ranking in a similar position.

My ML-06 audit found that staleness was a **MIXED** signal, so staleness is not enough by itself to justify a strong recommendation. Visibility is required, and CTR-vs-position provides additional evidence.

The baseline uses a simple point system:

### Visibility
- 300–2,999 impressions over 90 days → +1
- 3,000–29,999 → +2
- 30,000+ → +3

### Staleness
- 90–179 days since update → +1
- 180–364 days → +2
- 365+ days → +3

### CTR opportunity
- CTR below the median CTR of pages in the same position tier, with at least 250 impressions → +2

A page must have at least 300 impressions and either a staleness signal or a CTR opportunity to enter the review queue.

### Reason codes

Each page receives exactly one primary reason code:

- `STALE_VISIBLE_LOW_CTR` — stale, visible, and CTR is weak for its position
- `STALE_VISIBLE` — stale and visible
- `VISIBLE_LOW_CTR` — visible and CTR is weak for its position
- `NOT_PRIORITIZED` — does not meet the review rule

### Action labels

- `REVIEW_REFRESH_AND_CTR`
- `REVIEW_FOR_REFRESH`
- `REVIEW_CTR`
- `NO_ACTION`

The score is intentionally simple and uses no fitted weights. It is decision-support, not a prediction of Google's ranking algorithm.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

# Outcome used ONLY for retrospective evaluation.
# It is never used to calculate the baseline score.
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print(
    "Observed decline base rate:",
    f"{df['is_declining_label'].mean():.1%}"
)

# avg_position = 0 means position data is unavailable.
valid_position = (
    (df["avg_position"] > 0)
    & (df["impressions_90d"] >= 250)
)

# Typical CTR for each position tier.
expected_ctr_by_tier = (
    df.loc[valid_position]
    .groupby("position_tier", observed=True)["ctr"]
    .median()
)

print("\nMedian CTR by position tier:")
display(
    expected_ctr_by_tier
    .rename("median_ctr")
    .reset_index()
)

Dataset shape: (30000, 44)
Observed decline base rate: 54.2%

Median CTR by position tier:


,position_tier,median_ctr
0,deep,0.00
1,page_1,0.23
2,page_3_5,0.08
3,striking,0.17
4,top_3,0.19


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I convert the rule above into an integer score.

The score intentionally uses hand-written points rather than learned weights so that every recommendation can be explained directly.

`trend_pct`, `trend_direction`, and `is_declining_label` are not used anywhere in the score. IDs are also not predictive inputs.

The final queue is ranked first by baseline score and then by impressions, so ties favor pages with greater observed visibility.

In [3]:
queue = df.copy()

# ---------------------------------------------------------
# 1. VISIBILITY POINTS
# ---------------------------------------------------------

queue["visibility_points"] = np.select(
    [
        queue["impressions_90d"] >= 30000,
        queue["impressions_90d"] >= 3000,
        queue["impressions_90d"] >= 300,
    ],
    [
        3,
        2,
        1,
    ],
    default=0,
)


# ---------------------------------------------------------
# 2. STALENESS POINTS
# ---------------------------------------------------------

queue["staleness_points"] = np.select(
    [
        queue["days_since_last_update"] >= 365,
        queue["days_since_last_update"] >= 180,
        queue["days_since_last_update"] >= 90,
    ],
    [
        3,
        2,
        1,
    ],
    default=0,
)


# ---------------------------------------------------------
# 3. CTR VS POSITION
# ---------------------------------------------------------

queue["expected_ctr"] = (
    queue["position_tier"]
    .map(expected_ctr_by_tier)
    .astype(float)
)

queue["low_ctr_vs_position"] = (
    (queue["avg_position"] > 0)
    & (queue["impressions_90d"] >= 250)
    & queue["expected_ctr"].notna()
    & (queue["ctr"] < queue["expected_ctr"])
)

queue["ctr_points"] = (
    queue["low_ctr_vs_position"].astype(int) * 2
)


# ---------------------------------------------------------
# 4. ELIGIBILITY
# ---------------------------------------------------------

queue["eligible_for_review"] = (
    (queue["visibility_points"] > 0)
    & (
        (queue["staleness_points"] > 0)
        | (queue["ctr_points"] > 0)
    )
)


# ---------------------------------------------------------
# 5. FINAL SCORE
# ---------------------------------------------------------

queue["baseline_action_score"] = np.where(
    queue["eligible_for_review"],
    (
        queue["visibility_points"]
        + queue["staleness_points"]
        + queue["ctr_points"]
    ),
    0,
)


# ---------------------------------------------------------
# 6. ONE PRIMARY REASON CODE
# ---------------------------------------------------------

conditions = [
    queue["eligible_for_review"]
    & (queue["staleness_points"] > 0)
    & (queue["ctr_points"] > 0),

    queue["eligible_for_review"]
    & (queue["staleness_points"] > 0),

    queue["eligible_for_review"]
    & (queue["ctr_points"] > 0),
]

reason_codes = [
    "STALE_VISIBLE_LOW_CTR",
    "STALE_VISIBLE",
    "VISIBLE_LOW_CTR",
]

queue["reason_code"] = np.select(
    conditions,
    reason_codes,
    default="NOT_PRIORITIZED",
)


# ---------------------------------------------------------
# 7. ACTION LABEL
# ---------------------------------------------------------

action_labels = [
    "REVIEW_REFRESH_AND_CTR",
    "REVIEW_FOR_REFRESH",
    "REVIEW_CTR",
]

queue["action_label"] = np.select(
    conditions,
    action_labels,
    default="NO_ACTION",
)


# ---------------------------------------------------------
# 8. RANK
# ---------------------------------------------------------

queue = queue.sort_values(
    by=[
        "baseline_action_score",
        "impressions_90d",
        "days_since_last_update",
    ],
    ascending=[
        False,
        False,
        False,
    ],
).reset_index(drop=True)

queue["baseline_rank"] = np.arange(
    1,
    len(queue) + 1
)


print(
    "Pages prioritized:",
    int(queue["eligible_for_review"].sum())
)

print(
    "Share prioritized:",
    f"{queue['eligible_for_review'].mean():.1%}"
)


# ---------------------------------------------------------
# 9. WRITE CSV
# ---------------------------------------------------------

OUTPUT_PATH = Path(
    "../../work/outputs/baseline_action_score.csv"
)

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "position_tier",
    "expected_ctr",
    "low_ctr_vs_position",
]

queue[output_columns].to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nCSV written to:")
print(OUTPUT_PATH)

display(
    queue[output_columns].head(20)
)

# ---------------------------------------------------------
# RETROSPECTIVE EVALUATION
# Outcome is used here only, AFTER the queue was built.
# ---------------------------------------------------------

def precision_at_k(frame, k):
    return frame.head(k)["is_declining_label"].mean()


base_rate = queue["is_declining_label"].mean()
p20 = precision_at_k(queue, 20)
p50 = precision_at_k(queue, 50)

print("\nRETROSPECTIVE EVALUATION")
print(f"Base decline rate: {base_rate:.3f}")
print(f"Precision@20:      {p20:.3f}")
print(f"Precision@50:      {p50:.3f}")

Pages prioritized: 12500
Share prioritized: 41.7%

CSV written to:
../../work/outputs/baseline_action_score.csv


,content_id,client_id,baseline_rank,baseline_action_score,reason_code,action_label,impressions_90d,days_since_last_update,avg_position,ctr,position_tier,expected_ctr,low_ctr_vs_position
0,content_cf56e2e2e282,client_7f2253d7e2,1,7,STALE_VISIBLE_LOW_CTR,REVIEW_REFRESH_AND_CTR,61678,194,19.7,0.15,striking,0.17,True
1,content_5fe46e04994d,client_4e07408562,2,6,STALE_VISIBLE_LOW_CTR,REVIEW_REFRESH_AND_CTR,517715,104,4.2,0.14,page_1,0.23,True
2,content_cb112fce36be,client_19581e27de,3,6,STALE_VISIBLE_LOW_CTR,REVIEW_REFRESH_AND_CTR,309910,104,5.6,0.16,page_1,0.23,True
3,content_36ff89c8214e,client_19581e27de,4,6,STALE_VISIBLE_LOW_CTR,REVIEW_REFRESH_AND_CTR,295097,104,7.3,0.05,page_1,0.23,True
4,content_b28d1efd668f,client_6208ef0f77,5,6,STALE_VISIBLE_LOW_CTR,REVIEW_REFRESH_AND_CTR,286608,104,26.2,0.06,page_3_5,0.08,True
5,content_813e88069237,client_6208ef0f77,6,6,STALE_VISIBLE_LOW_CTR,REVIEW_REFRESH_AND_CTR,233561,104,26.2,0.06,page_3_5,0.08,True
6,content_c8e9d6ab9013,client_19581e27de,7,6,STALE_VISIBLE_LOW_CTR,REVIEW_REFRESH_AND_CTR,208678,104,9.7,0.00,page_1,0.23,True
7,content_a7427266c305,client_19581e27de,8,6,STALE_VISIBLE_LOW_CTR,REVIEW_REFRESH_AND_CTR,201111,104,5.7,0.11,page_1,0.23,True
8,content_33b4dceecad1,client_19581e27de,9,6,STALE_VISIBLE_LOW_CTR,REVIEW_REFRESH_AND_CTR,181574,104,6.2,0.16,page_1,0.23,True
9,content_91652435f57a,client_19581e27de,10,6,STALE_VISIBLE_LOW_CTR,REVIEW_REFRESH_AND_CTR,159590,104,7.8,0.06,page_1,0.23,True



RETROSPECTIVE EVALUATION
Base decline rate: 0.542
Precision@20:      0.650
Precision@50:      0.520


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I inspect the top 20 recommendations rather than assuming that a high score means the recommendation is correct.

For each page I record:

- the proposed action,
- the primary reason code,
- a simple confidence note,
- why the page reached the top of the queue,
- and what evidence could make the recommendation wrong.

The decline outcome is used only here for retrospective review. It was not used to construct or rank the queue.

In [4]:
top20 = queue.head(20).copy()


def confidence_note(row):
    if row["baseline_action_score"] >= 7:
        return "HIGH"
    elif row["baseline_action_score"] >= 5:
        return "MEDIUM"
    else:
        return "LOW"


def why_it_is_here(row):
    reasons = [
        f"{int(row['impressions_90d']):,} impressions/90d"
    ]

    if row["staleness_points"] > 0:
        reasons.append(
            f"{int(row['days_since_last_update'])} days since update"
        )

    if row["low_ctr_vs_position"]:
        reasons.append(
            "CTR below position-tier median"
        )

    return "; ".join(reasons)


def what_would_make_it_wrong(row):

    # Retrospective false positive
    if row["is_declining_label"] == 0:
        return (
            "Observed outcome was not declining; "
            "the baseline may be over-prioritizing this page."
        )

    if (
        row["staleness_points"] > 0
        and row["low_ctr_vs_position"]
    ):
        return (
            "The page may be intentionally evergreen, "
            "or low CTR may reflect SERP/query intent rather "
            "than a content problem."
        )

    if row["staleness_points"] > 0:
        return (
            "Age alone may not be actionable; the page may be "
            "evergreen and a refresh may not improve performance."
        )

    return (
        "Low CTR may be normal for this query mix or SERP, "
        "so changing the content may not improve clicks."
    )


top20_review = pd.DataFrame({
    "rank": top20["baseline_rank"],
    "action": top20["action_label"],
    "reason_code": top20["reason_code"],
    "score": top20["baseline_action_score"],
    "confidence": top20.apply(
        confidence_note,
        axis=1
    ),
    "why_it_is_here": top20.apply(
        why_it_is_here,
        axis=1
    ),
    "what_would_make_it_wrong": top20.apply(
        what_would_make_it_wrong,
        axis=1
    ),
})

display(top20_review)

print(
    "\nObserved declining pages in top 20:",
    int(top20["is_declining_label"].sum()),
    "/ 20"
)

,rank,action,reason_code,score,confidence,why_it_is_here,what_would_make_it_wrong
0,1,REVIEW_REFRESH_AND_CTR,STALE_VISIBLE_LOW_CTR,7,HIGH,"61,678 impressions/90d; 194 days since update;...","The page may be intentionally evergreen, or lo..."
1,2,REVIEW_REFRESH_AND_CTR,STALE_VISIBLE_LOW_CTR,6,MEDIUM,"517,715 impressions/90d; 104 days since update...","The page may be intentionally evergreen, or lo..."
2,3,REVIEW_REFRESH_AND_CTR,STALE_VISIBLE_LOW_CTR,6,MEDIUM,"309,910 impressions/90d; 104 days since update...","The page may be intentionally evergreen, or lo..."
3,4,REVIEW_REFRESH_AND_CTR,STALE_VISIBLE_LOW_CTR,6,MEDIUM,"295,097 impressions/90d; 104 days since update...",Observed outcome was not declining; the baseli...
4,5,REVIEW_REFRESH_AND_CTR,STALE_VISIBLE_LOW_CTR,6,MEDIUM,"286,608 impressions/90d; 104 days since update...",Observed outcome was not declining; the baseli...
5,6,REVIEW_REFRESH_AND_CTR,STALE_VISIBLE_LOW_CTR,6,MEDIUM,"233,561 impressions/90d; 104 days since update...","The page may be intentionally evergreen, or lo..."
6,7,REVIEW_REFRESH_AND_CTR,STALE_VISIBLE_LOW_CTR,6,MEDIUM,"208,678 impressions/90d; 104 days since update...","The page may be intentionally evergreen, or lo..."
7,8,REVIEW_REFRESH_AND_CTR,STALE_VISIBLE_LOW_CTR,6,MEDIUM,"201,111 impressions/90d; 104 days since update...",Observed outcome was not declining; the baseli...
8,9,REVIEW_REFRESH_AND_CTR,STALE_VISIBLE_LOW_CTR,6,MEDIUM,"181,574 impressions/90d; 104 days since update...",Observed outcome was not declining; the baseli...
9,10,REVIEW_REFRESH_AND_CTR,STALE_VISIBLE_LOW_CTR,6,MEDIUM,"159,590 impressions/90d; 104 days since update...",Observed outcome was not declining; the baseli...



Observed declining pages in top 20: 13 / 20


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

A high baseline score does not prove that a page should be changed.

I treat top-ranked pages that did not show the observed decline outcome as weak picks. These cases are useful because they reveal where the hand-written rule may be too broad.

The score uses only observable content/performance signals:

- `impressions_90d`
- `days_since_last_update`
- `ctr`
- `avg_position`
- `position_tier`

It does not use `trend_pct`, `trend_direction`, or `is_declining_label` as scoring inputs. Those fields contain outcome information and are used only after ranking for retrospective evaluation.

No product flag, client identifier, content identifier, URL, private query, or future-window field is used as a scoring feature.

In [5]:
# ---------------------------------------------------------
# WEAK PICKS
# ---------------------------------------------------------

weak_picks = top20[
    top20["is_declining_label"] == 0
].copy()

print(
    "Weak / observed false-positive picks in top 20:",
    len(weak_picks)
)

if len(weak_picks) > 0:

    display(
        weak_picks[
            [
                "baseline_rank",
                "baseline_action_score",
                "reason_code",
                "impressions_90d",
                "days_since_last_update",
                "avg_position",
                "ctr",
            ]
        ]
    )

else:

    print(
        "No observed false positives in the top 20. "
        "Showing the weakest-scoring top recommendations instead."
    )

    display(
        top20.sort_values(
            by=[
                "baseline_action_score",
                "impressions_90d",
            ],
            ascending=[
                True,
                True,
            ],
        )[
            [
                "baseline_rank",
                "baseline_action_score",
                "reason_code",
                "impressions_90d",
                "days_since_last_update",
                "avg_position",
                "ctr",
            ]
        ].head(3)
    )

# ---------------------------------------------------------
# LEAKAGE CHECK
# ---------------------------------------------------------

RULE_INPUTS = {
    "impressions_90d",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "position_tier",
}

FORBIDDEN_OUTCOME_INPUTS = {
    "trend_pct",
    "trend_direction",
    "is_declining_label",
}

PRODUCT_FLAGS = {
    "health_score",
    "quick_win",
    "needs_attention",
}

assert RULE_INPUTS.isdisjoint(
    FORBIDDEN_OUTCOME_INPUTS
)

assert RULE_INPUTS.isdisjoint(
    PRODUCT_FLAGS
)

assert "client_id" not in RULE_INPUTS
assert "content_id" not in RULE_INPUTS

print("\nLeakage check: PASS")
print(
    "Outcome-derived fields used in scoring: NONE"
)
print(
    "Product flags used in scoring: NONE"
)
print(
    "IDs used as predictive features: NONE"
)


# ---------------------------------------------------------
# FINAL OUTPUT CHECK
# ---------------------------------------------------------

assert len(queue) == len(df)
assert queue["baseline_rank"].is_unique
assert OUTPUT_PATH.exists()

print("\nFinal checks: PASS")
print("Queue rows:", len(queue))
print("CSV exists:", OUTPUT_PATH.exists())

Weak / observed false-positive picks in top 20: 7


,baseline_rank,baseline_action_score,reason_code,impressions_90d,days_since_last_update,avg_position,ctr
3,4,6,STALE_VISIBLE_LOW_CTR,295097,104,7.3,0.05
4,5,6,STALE_VISIBLE_LOW_CTR,286608,104,26.2,0.06
7,8,6,STALE_VISIBLE_LOW_CTR,201111,104,5.7,0.11
8,9,6,STALE_VISIBLE_LOW_CTR,181574,104,6.2,0.16
9,10,6,STALE_VISIBLE_LOW_CTR,159590,104,7.8,0.06
15,16,6,STALE_VISIBLE_LOW_CTR,130932,104,40.1,0.04
19,20,6,STALE_VISIBLE_LOW_CTR,128068,104,2.2,0.01



Leakage check: PASS
Outcome-derived fields used in scoring: NONE
Product flags used in scoring: NONE
IDs used as predictive features: NONE

Final checks: PASS
Queue rows: 30000
CSV exists: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.